### 체크리스트

- 한 행의 단위 (head)
- 결측이 어디 있고, 왜 있을까 (info)
- 데이터의 dtype과 실제 데이터 내용 비교
- 분포 파악 (describe) — unique=총행수면 PK, unique가 작으면 카테고리형

#### olist_orders_dataset.csv

In [ ]:
import pandas as pd

In [ ]:
orders = pd.read_csv('../data/olist_orders_dataset.csv')

In [6]:
# shape 확인

print(orders.shape)
print(orders.columns)
print(orders.head(5))

(99441, 8)
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59

데이터프레임의 한 행은 주문 1건의 라이프사이클을 나타낸다. key 컬럼은 order_id이며, 구매>승인>배송출발>배송완료>예상배송 순서로 진행

In [7]:
print(orders.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
None


- order_approved_at 결측: 구매는 하였으나 승인되지 않은 주문 건 존재
- order_delivered_carrier_date, customer_date 결측: 아직 배송이 시작되지 않았거나, 배송 중인 주문 건 존재 
- dtypes: object: 추후 배송 리드타임 계산 등, 날짜 계산을 시도한다면 datetime변환 필요

In [8]:
print(orders.describe())

                                order_id                       customer_id  \
count                              99441                             99441   
unique                             99441                             99441   
top     e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
freq                                   1                                 1   

       order_status order_purchase_timestamp    order_approved_at  \
count         99441                    99441                99281   
unique            8                    98875                90733   
top       delivered      2018-08-02 12:05:26  2018-02-27 04:31:10   
freq          96478                        3                    9   

       order_delivered_carrier_date order_delivered_customer_date  \
count                         97658                         96476   
unique                        81018                         95664   
top             2018-05-09 15:48:00           2018-05-08

- order_id: primary key
- customer_id: 모든 행이 unique -> 고객 데이터 중복 없음 (즉, 재구매 고객 없음)
- order_status: 주문 상태값은 8종류, 현재 '배송중' 상태가 가장 많음 (약 97%로 추산)
- order_estimated_delivery_date: unique 행이 459개 -> 판매자는 배송을 특정 일자에 몰아서 진행함.

##### 필터링해서 심층 분석

In [9]:
# 주요 컬럼으로 축소
cols = ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']
orders = orders[cols]

In [10]:
# 주문 상태 8종 파악
pd.DataFrame({
    'count': orders['order_status'].value_counts(),
    '%': orders['order_status'].value_counts(normalize=True).mul(100).round(1)
})

,count,%
order_status,,
delivered,96478,97.0
shipped,1107,1.1
canceled,625,0.6
unavailable,609,0.6
invoiced,314,0.3
processing,301,0.3
created,5,0.0
approved,2,0.0


- 주문 라이프사이클:
1) 완료시: created → approved → processing → invoiced → shipped → delivered
2) 취소시: created → approved → processing → canceled / unavailable
- 데이터 상 주문 건은 대부분 거래 완료 (배송 완료)

In [11]:
# 의미 있는 데이터 (구매완료 건) 솎아내기

delivered = orders[orders['order_status'] == 'delivered']
delivered.shape

(96478, 4)

#### olist_order_items_dataset.csv

In [24]:
items = pd.read_csv('../data/olist_order_items_dataset.csv')

print("====shape====\n",items.shape)
print("\n====head====\n",items.head(3))
print("\n====columns====\n",items.columns)

====shape====
 (112650, 7)

====head====
                            order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   

   shipping_limit_date  price  freight_value  
0  2017-09-19 09:45:35   58.9          13.29  
1  2017-05-03 11:05:13  239.9          19.93  
2  2018-01-18 14:48:30  199.0          17.87  

====columns====
 Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')


In [26]:
print(items.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB
None


- 결측치 없음
- PK: order_id

In [27]:
print(items.describe())


       order_item_id          price  freight_value
count  112650.000000  112650.000000  112650.000000
mean        1.197834     120.653739      19.990320
std         0.705124     183.633928      15.806405
min         1.000000       0.850000       0.000000
25%         1.000000      39.900000      13.080000
50%         1.000000      74.990000      16.260000
75%         1.000000     134.900000      21.150000
max        21.000000    6735.000000     409.680000


- order_item_id: 한 주문 안에서 몇 번째 아이템인지 나타내는 데이터. 즉, 주문(고객)의 75% 이상이 아이템 1개씩만 구매함 (장바구니 사이즈 1개)
- price: 평균 120 > 중앙값 75 = 소수의 고가 상품이 객단가를 높인다.

#### olist_products_dataset.csv

In [29]:
products = pd.read_csv('../data/olist_products_dataset.csv')

print("====shape====\n",products.shape)
print("\n====head====\n",products.head(3))
print("\n====columns====\n",products.columns)

====shape====
 (32951, 9)

====head====
                          product_id product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5            perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                 artes   
2  96bd76ec8810374ed1b65e291975717f         esporte_lazer   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                 40.0                       287.0                 1.0   
1                 44.0                       276.0                 1.0   
2                 46.0                       250.0                 1.0   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0             225.0               16.0               10.0              14.0  
1            1000.0               30.0               18.0              20.0  
2             154.0               18.0                9.0              15.0  

====columns====
 Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_de

In [30]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


- PK: product_id
- category name에 결측치 확인 필요

In [35]:
products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [53]:
# 카테고리 id 종류 파악

pd.DataFrame({
    'count': products['product_category_name'].value_counts(),
    '%': products['product_category_name'].value_counts(normalize=True).mul(100).round(1)
})


,count,%
product_category_name,,
cama_mesa_banho,3029,9.4
esporte_lazer,2867,8.9
moveis_decoracao,2657,8.2
beleza_saude,2444,7.6
utilidades_domesticas,2335,7.2
...,...,...
fashion_roupa_infanto_juvenil,5,0.0
casa_conforto_2,5,0.0
pc_gamer,3,0.0


In [50]:
# 카테고리id 결측치 데이터 확인
products[products['product_category_name'].isna()].info()


<class 'pandas.core.frame.DataFrame'>
Index: 610 entries, 105 to 32852
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  610 non-null    object 
 1   product_category_name       0 non-null      object 
 2   product_name_lenght         0 non-null      float64
 3   product_description_lenght  0 non-null      float64
 4   product_photos_qty          0 non-null      float64
 5   product_weight_g            609 non-null    float64
 6   product_length_cm           609 non-null    float64
 7   product_height_cm           609 non-null    float64
 8   product_width_cm            609 non-null    float64
dtypes: float64(7), object(2)
memory usage: 47.7+ KB


- 카테고리는 총 73개
- 카테고리가 Null 인 데이터는 610 행으로, 전체 데이터의 2% 이하이므로 이후 분석에서는 drop

#### product_category_name_translation.csv

In [54]:
translation = pd.read_csv('../data/product_category_name_translation.csv')


print("====shape====\n",translation.shape)
print("\n====head====\n",translation.head(3))
print("\n====columns====\n",translation.columns)

====shape====
 (71, 2)

====head====
     product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto

====columns====
 Index(['product_category_name', 'product_category_name_english'], dtype='object')


In [55]:
translation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [57]:
pd.DataFrame({
    'count': translation['product_category_name'].value_counts(),
    '%': translation['product_category_name'].value_counts(normalize=True).mul(100).round(1)
})


,count,%
product_category_name,,
beleza_saude,1,1.4
informatica_acessorios,1,1.4
automotivo,1,1.4
cama_mesa_banho,1,1.4
moveis_decoracao,1,1.4
...,...,...
flores,1,1.4
artes_e_artesanato,1,1.4
fraldas_higiene,1,1.4


In [56]:
translation.describe()

,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


- 결측 없음
- 카테고리가 총 71개로, products 테이블 데이터와 갯수 불일치